# Unidad 3 – Sesión 1
## Limpieza y tratamiento inicial de datos

**Curso:** Minería de Datos  
**Fase CRISP-DM:** Preparación de los Datos  

### Propósito
Al finalizar la sesión, el estudiante será capaz de:
- identificar problemas que requieren preparación;
- tratar nulos, duplicados, inconsistencias, valores inválidos y outliers;
- justificar cada decisión;
- validar el resultado antes y después de transformar los datos.

> **Idea clave:** En la Unidad 2 detectamos los problemas. En la Unidad 3 comenzamos a resolverlos.

# 1. Dataset de ejemplo

Usaremos un dataset pequeño con problemas intencionales de calidad.

In [ ]:
import pandas as pd
import numpy as np

data = {
    "cliente_id": [1, 2, 3, 3, 4, 5, 6, 7],
    "edad": [25, 32, None, 40, 250, 28, -5, 36],
    "ciudad": ["Lima", "LIMA", "lima", "Cusco", "Cusco", None, "LIMA ", "Arequipa"],
    "ingreso": [1500, 2300, None, 1800, 50000, 1700, 1600, 1750],
    "estado": ["Si", "Sí", "SI", "No", "NO", "yes", "NO", "SI"]
}

df = pd.DataFrame(data)
df

### Iniciamos....
1. ¿Qué problemas detectas?
2. ¿Qué variables presentan nulos?
3. ¿Hay categorías inconsistentes?
4. ¿Hay valores inválidos?
5. ¿Hay posibles outliers?
6. ¿Existen posibles duplicados?

# 2. Detectar vs preparar

En la Unidad 2 diagnosticábamos:

```python
df.isnull().sum()
df.duplicated().sum()
df["ciudad"].value_counts()
df.describe()
```

En la Unidad 3 tomamos decisiones y transformamos:

```python
df["ciudad"] = df["ciudad"].str.strip().str.upper()
```

La secuencia será:

**Problema → Evidencia → Decisión → Transformación → Validación**

In [ ]:
plan_ejemplo = pd.DataFrame({
    "Problema": [
        "Categoría inconsistente",
        "Edad nula",
        "Posible duplicado",
        "Edad inválida"
    ],
    "Evidencia": [
        "Lima / LIMA / lima / LIMA ",
        "Existe al menos un valor nulo",
        "cliente_id = 3 aparece más de una vez",
        "Edad = 250 y Edad = -5"
    ],
    "Decision": [
        "Estandarizar",
        "Imputar",
        "Revisar llave lógica",
        "Convertir a nulo"
    ],
    "Transformacion": [
        "strip() + upper()",
        "Mediana",
        "duplicated(subset=...)",
        "Regla de rango"
    ],
    "Validacion": [
        "value_counts()",
        "isnull().sum()",
        "Volver a contar duplicados",
        "describe()"
    ]
})
plan_ejemplo

In [ ]:
df.duplicated().sum()

# 3. Tratamiento de valores nulos

In [ ]:
df.isnull().sum()

In [ ]:
(df.isnull().mean() * 100).round(2)

## Eliminar filas
Puede utilizarse cuando los nulos son pocos y la pérdida de registros no afecta de forma importante el análisis.

In [ ]:
df_sin_nulos = df.dropna()
df_sin_nulos

## Imputación numérica
La mediana suele ser más robusta cuando existen valores extremos.

In [ ]:
df_imputado = df.copy()
df_imputado["ingreso"] = df_imputado["ingreso"].fillna(df_imputado["ingreso"].median())
df_imputado["ingreso"].isnull().sum()

## Imputación categórica
Se puede usar la moda o una categoría explícita como `"DESCONOCIDO"`.

In [ ]:
df_categorico = df.copy()
df_categorico["ciudad"] = df_categorico["ciudad"].fillna("DESCONOCIDO")
df_categorico["ciudad"].value_counts(dropna=False)

¿Qué ventaja tiene usar `"DESCONOCIDO"` en vez de reemplazar automáticamente por la categoría más frecuente?

# 4. Duplicados

In [ ]:
df.duplicated().sum()

In [ ]:
df[df.duplicated(subset=["cliente_id"], keep=False)]

> Un identificador repetido no implica automáticamente un error. Primero debe verificarse si la granularidad exige una sola fila por cliente.

# 5. Consistencia de variables categóricas

In [ ]:
df["ciudad"].value_counts(dropna=False)

In [ ]:
df_limpio = df.copy()

df_limpio["ciudad"] = (
    df_limpio["ciudad"]
    .astype("string")
    .str.strip()
    .str.upper()
)

df_limpio["ciudad"].value_counts(dropna=False)

In [ ]:
df_limpio["estado"] = (
    df_limpio["estado"]
    .astype("string")
    .str.strip()
    .str.upper()
    .replace({
        "SÍ": "SI",
        "YES": "SI"
    })
)

df_limpio["estado"].value_counts(dropna=False)

# 6. Valores inválidos

In [ ]:
df["edad"].describe()

In [ ]:
df_validacion = df.copy()

df_validacion.loc[
    (df_validacion["edad"] < 0) |
    (df_validacion["edad"] > 100),
    "edad"
] = np.nan

df_validacion["edad"].describe()

In [ ]:
df_validacion["edad"] = df_validacion["edad"].fillna(
    df_validacion["edad"].median()
)

df_validacion["edad"].isnull().sum()

# 7. Outliers

> **Un outlier no significa automáticamente que exista un error.**

In [ ]:
Q1 = df["ingreso"].quantile(0.25)
Q3 = df["ingreso"].quantile(0.75)
IQR = Q3 - Q1

limite_inferior = Q1 - 1.5 * IQR
limite_superior = Q3 + 1.5 * IQR

print("Q1:", Q1)
print("Q3:", Q3)
print("IQR:", IQR)
print("Límite inferior:", limite_inferior)
print("Límite superior:", limite_superior)

In [ ]:
outliers_ingreso = df[
    (df["ingreso"] < limite_inferior) |
    (df["ingreso"] > limite_superior)
]

outliers_ingreso

### Posibles decisiones
- Conservar
- Corregir
- Transformar
- Excluir
- Analizar por separado

La decisión siempre debe estar justificada.

# 8. Validación antes y después

In [ ]:
df_validado = df.copy()

print("ANTES")
print(df_validado["ciudad"].value_counts(dropna=False))

df_validado["ciudad"] = (
    df_validado["ciudad"]
    .astype("string")
    .str.strip()
    .str.upper()
)

print("\nDESPUÉS")
print(df_validado["ciudad"].value_counts(dropna=False))

In [ ]:
print("Nulos en edad ANTES:", df["edad"].isnull().sum())

df_validado["edad"] = df["edad"].copy()

df_validado.loc[
    (df_validado["edad"] < 0) |
    (df_validado["edad"] > 100),
    "edad"
] = np.nan

df_validado["edad"] = df_validado["edad"].fillna(
    df_validado["edad"].median()
)

print("Nulos en edad DESPUÉS:", df_validado["edad"].isnull().sum())

# 9. Actividad práctica

Carga tu propio dataset.

In [ ]:
from google.colab import files
uploaded = files.upload()

In [ ]:
# Para CSV:
df_proyecto = pd.read_csv("LI-Small_Trans.csv")

# Para Excel:
# df_proyecto = pd.read_excel("archivo.xlsx")

- 2 variables con nulos;


In [ ]:
nulos = df_proyecto.isnull().sum()
nulos[nulos > 0]

In [ ]:
df_proyecto[df_proyecto.duplicated(keep=False)]

In [ ]:
# Revisar categorías y detectar diferencias de escritura o espacios
columnas = ["Receiving Currency", "Payment Currency", "Payment Format"]

for columna in columnas:
    print(f"\n{columna}:")
    print(df_proyecto[columna].value_counts(dropna=False))

In [ ]:
# Resumen estadístico de los montos
df_proyecto[["Amount Received", "Amount Paid"]].describe()

In [ ]:
df_proyecto.groupby("Receiving Currency")["Amount Received"].describe()

In [ ]:
df_proyecto[
    df_proyecto["Is Laundering"].notna() &
    ~df_proyecto["Is Laundering"].isin([0, 1])
]

In [ ]:
montos = df_proyecto.loc[
    df_proyecto["Receiving Currency"] == "US Dollar",
    "Amount Received"
]

Q1 = montos.quantile(0.25)
Q3 = montos.quantile(0.75)
IQR = Q3 - Q1

limite_inferior = Q1 - 1.5 * IQR
limite_superior = Q3 + 1.5 * IQR

outliers = df_proyecto.loc[
    montos[
        (montos < limite_inferior) |
        (montos > limite_superior)
    ].index
]

outliers.head()

Identifica como mínimo:
- 2 variables con nulos;
- 1 posible duplicado;
- 1 problema de consistencia;
- 1 valor inválido;
- 1 posible outlier.

In [ ]:
plan_preparacion = pd.DataFrame({
    "Variable": [
        "Todas las variables",
        "Transacción completa",
        "Receiving Currency / Payment Currency / Payment Format",
        "Amount Received / Amount Paid",
        "Is Laundering",
        "Amount Received"
    ],
"Problema": [
    "No hay",
    "Posibles duplicados",
    "No hay",
    "No hay",
    "No hay",
    "Posible outlier"
],
    "Evidencia": [
        "Todas las variables tienen cero nulos",
        "Los registros 676694 y 676696 coinciden en todas sus columnas",
        "Las categorías revisadas no presentan diferencias de escritura",
        "No hay montos negativos ni ceros",
        "Solo contiene 0 y 1",
        "Registro 0: 3 195 403 USD supera el límite IQR de 12 377.16 USD"
    ],
    "Decision": [
        "Conservar",
        "Revisar antes de eliminar",
        "Conservar",
        "Conservar",
        "Conservar",
        "Conservar y revisar"
    ],
    "Transformacion": [
        "Ninguna",
        "Pendiente: marcar posibles duplicados",
        "Ninguna",
        "Ninguna",
        "Ninguna",
        "Pendiente: marcar el posible outlier"
    ],
    "Justificacion": [
        "No hay nulos que tratar",
        "Podrían ser operaciones diferentes realizadas en el mismo minuto",
        "No se detectaron inconsistencias que corregir",
        "No se detectaron incumplimientos de las reglas revisadas",
        "Los valores pertenecen al dominio permitido",
        "Ser un valor extremo no demuestra que sea un error"
    ]
})

plan_preparacion

### Decisiones de preparación

1. **Valores nulos**
   - **Problema:** No se encontraron valores nulos.
   - **Evidencia:** El conteo de nulos fue cero en todas las variables.
   - **Decisión:** Conservar los datos sin imputar.
   - **Transformación aplicada:** Ninguna.
   - **Justificación:** No existen valores faltantes que requieran tratamiento.

2. **Posibles duplicados**
   - **Problema:** Se encontraron filas idénticas.
   - **Evidencia:** Los registros 676694 y 676696 coinciden en todas sus columnas.
   - **Decisión:** Conservarlos para revisión antes de eliminarlos.
   - **Transformación aplicada:** Ninguna; se identificaron mediante `duplicated(keep=False)`.
   - **Justificación:** Podrían ser operaciones diferentes realizadas en el mismo minuto, ya que no existe un identificador único de transacción.

3. **Consistencia categórica**
   - **Problema:** No se detectaron inconsistencias en las categorías revisadas.
   - **Evidencia:** Las monedas y los formatos de pago no mostraron diferencias de escritura.
   - **Decisión:** Conservar las categorías.
   - **Transformación aplicada:** Ninguna.
   - **Justificación:** No se encontraron inconsistencias que requieran corrección.

4. **Valores inválidos**
   - **Problema:** No se detectaron valores inválidos con las reglas revisadas.
   - **Evidencia:** Los montos son positivos y `Is Laundering` solo contiene 0 y 1.
   - **Decisión:** Conservar los valores.
   - **Transformación aplicada:** Ninguna.
   - **Justificación:** Los valores cumplen las reglas de validación revisadas.

5. **Posible outlier**
   - **Problema:** Se identificó un monto recibido estadísticamente extremo.
   - **Evidencia:** El registro 0 contiene 3 195 403 USD y supera el límite superior IQR de aproximadamente 12 377.16 USD.
   - **Decisión:** Conservarlo para revisión.
   - **Transformación aplicada:** Ninguna; se identificó mediante la regla IQR para transacciones en dólares.
   - **Justificación:** Un monto extremo no implica necesariamente un error y puede ser relevante para detectar operaciones inusuales.

# 10. Matriz final de preparación

In [ ]:
matriz_final = pd.DataFrame({
    "Variable": [
        "Todas las variables",
        "Transacción completa",
        "Receiving Currency / Payment Currency / Payment Format",
        "Amount Received / Amount Paid / Is Laundering",
        "Amount Received"
    ],
    "Problema": [
        "No hay nulos",
        "Posibles duplicados",
        "No se detectaron inconsistencias",
        "No se detectaron valores inválidos con las reglas revisadas",
        "Posible outlier"
    ],
    "Evidencia": [
        "Cero valores nulos en todas las columnas",
        "Los registros 676694 y 676696 coinciden en todas sus columnas",
        "No se observaron diferencias de escritura en las categorías",
        "Montos positivos y etiquetas con valores 0 y 1",
        "Registro 0: 3 195 403 USD supera el límite IQR de 12 377.16 USD"
    ],
    "Decision": [
        "Conservar sin imputar",
        "Conservar para revisión",
        "Conservar las categorías",
        "Conservar los valores",
        "Conservar para revisión"
    ],
    "Transformacion": [
        "Ninguna",
        "Ninguna; detección con duplicated",
        "Ninguna",
        "Ninguna",
        "Ninguna; detección mediante IQR para US Dollar"
    ],
    "Resultado": [
        "Datos conservados sin imputación",
        "Posibles duplicados identificados y conservados",
        "Categorías conservadas",
        "Valores conservados al cumplir las reglas revisadas",
        "Posible outlier identificado y conservado"
    ]
})

matriz_final

# 11. Cierre

1. **¿Eliminar nulos siempre mejora la calidad?**  
   No. Puede provocar la pérdida de información importante e introducir sesgos. La decisión depende de la cantidad de nulos y de su causa.

2. **¿Cuándo usarías media y cuándo mediana?**  
   Usaría la media cuando la distribución sea aproximadamente simétrica y no tenga valores extremos importantes. Usaría la mediana cuando haya asimetría o valores extremos.

3. **¿Un outlier debe eliminarse automáticamente?**  
   No. Puede representar un caso real y relevante, como una transacción inusual. Primero debe investigarse.

4. **¿Dos filas iguales siempre son duplicados?**  
   Son duplicados en sus valores, pero no necesariamente representan la misma operación. En este dataset podrían existir transacciones distintas realizadas en el mismo minuto.

5. **¿Qué riesgo existe al imputar demasiados valores?**  
   Puede reducir artificialmente la variabilidad, introducir sesgos y alterar las relaciones entre variables. En transacciones, imputar montos también podría inventar información financiera.

6. **¿Por qué debemos validar después de limpiar?**  
   Para comprobar que los cambios resolvieron los problemas detectados sin introducir errores ni perder información relevante.

7. **¿Qué diferencia existe entre valor inválido y outlier?**  
   Un valor inválido incumple una regla del dominio o del formato. Un outlier es estadísticamente atípico y puede ser válido o inválido; debe revisarse antes de decidir su tratamiento.

8. **¿Qué ocurriría si estandarizamos incorrectamente una categoría?**  
   Podríamos unir categorías con significados diferentes, perder información y distorsionar los resultados del análisis.

# 12. Conclusión

> **En Comprensión de los Datos identificamos los problemas. En Preparación de los Datos tomamos decisiones para transformar el dataset de manera justificada, reproducible y verificable.**

### Producto esperado
- diagnóstico;
- evidencia;
- decisión;
- transformación;
- validación antes/después;
- matriz final de preparación.